# Qiskit Fall Fest 2026, Session 4: HPC and quantum machine learning

Thu, Nov 19, 2026, 5:00 to 6:00 PM ET, on Zoom

Covers: Tensor networks, GPUs, quantum-centric supercomputing, kernels, and classifiers.

This is the fourth of five sessions this week. Tonight has three parts.

- Seminar: where simulation runs out, and what a kernel is.
- Lab: a simulator scaling test, then a quantum kernel classifier.
- Challenge 4 opens at 6:00 PM. Every earlier challenge stays open until Sun, Nov 22 at 11:59 PM.

Everything tonight runs on simulators. A simulator is a program that imitates a quantum computer on an ordinary computer. Nothing in this notebook needs an account, and no cell sends anything to a real quantum computer.

The words in the title and in the "Covers" line, and a few that come with them, in a sentence or two each. HPC is short for high-performance computing: solving large problems on large computers. Machine learning means letting a program find a rule from examples, where otherwise a person would write the rule by hand, and quantum machine learning hands part of that work to a quantum computer. A qubit is the quantum version of a bit, and the state of a set of qubits is their full description: everything you need in order to predict what a measurement will give. A tensor network is a way to store the state of many qubits as many small tables of numbers that are linked to each other, in place of one enormous list. Tonight's tensor network is the simplest kind, a chain. A GPU, short for graphics processing unit, is a chip that does the same arithmetic on very many numbers at once. Quantum-centric supercomputing is a way of working in which a quantum computer and a large ordinary computer share one computation. A kernel is a function that takes two data points and returns one number that says how alike they are. It has nothing to do with the Jupyter kernel that the Setup section mentions. A classifier is a program that sorts data points into groups. Qiskit is the open-source Python library, developed by IBM, that you use all week to write quantum programs, which are called circuits, and to run them.

This notebook stands on its own. If you missed the earlier sessions, start here anyway: Part 1 opens with the few ideas from Monday that tonight needs, and every other term is explained where it first appears.

## What you will be able to do

- Work out how much memory a state vector of $n$ qubits needs, and see where the computer that runs this notebook gives up. A state vector is the full list of numbers that describes the state of the qubits.
- Say why a matrix product state can store a GHZ state on 60 qubits, and when that trick stops working. A matrix product state is the simplest tensor network. A GHZ state is a state in which all the qubits are tied together, and Part 1 builds it.
- Turn a data point into a quantum state with a feature map, the circuit that takes in the data, and compute a quantum kernel.
- Train and test a kernel classifier, and compare it honestly with a classical one. Classical means ordinary, without anything quantum.
- Explain how the parameter-shift rule gives the gradient of a circuit: the slopes that tell a training program how to adjust the circuit.

## How to use this notebook

- Run the cells in order, from the top. A cell is one box of text or code. To run a code cell, click it and press Shift+Enter.
- Part 1 follows tonight's slides. Each idea has a short recap, and often a small cell you can run to see the idea at work.
- Part 2 is the lab. The code under "Lab" is complete. Run it, read the output, then change it and run it again.
- After each lab there is a "Your turn" exercise. Replace the line `# YOUR CODE HERE` with your own code, run the cell, then run the check cell below it. A check that fails stops with a red error message. Its last line is the hint. Run all (Colab: Runtime menu; Jupyter: Run menu) stops at the first check whose exercise you have not done yet. Do the exercise there, or run the cells after it one by one.
- Everything runs on a simulator, on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). This notebook has no hardware cell, a cell that can send work to a real quantum computer, so leave the switches in the Setup section as they are. The Setup section is the same in all five notebooks; tonight no cell uses the simulated processor `fake_fez` that the setup check prints. Every circuit runs on an ideal simulator, without noise.
- Use this notebook during the session and again afterwards on your own.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## Join the Discord and register

*Slide 2.*

- **Discord.** Join the server at https://discord.gg/FBNzRcxMr. Questions between sessions go there.
- **Registration.** Not registered yet? Fill in the form at https://forms.gle/NC9o1iRk3WggLuTH7.

On the slide the two links are QR codes, small square patterns that a phone camera reads as a link. Discord is the chat service where the club answers questions.

## New tonight?

*Slide 4.*

- **IBM Quantum Platform.** quantum.cloud.ibm.com. Open Plan instance and an API key. This is where your circuits run.
- **qBraid, the second option.** account.qbraid.com. Sign up with the email you registered with. Access is sent there.
- **Step by step.** The Accounts section of the Session 1 notebook, on the event page.

Tonight's lab needs no account. Open tonight's notebook and run the first cell.

You need an account only to run on a real quantum computer. Nothing in tonight's notebook does that. It has no hardware cell, which is a cell that can send work to a real quantum computer, so leave the three switches in the Setup section as they are. You can do the whole lab tonight and set up an account later.

Some words from the first card. IBM Quantum Platform is the website where you manage your IBM Quantum account. An API key is a secret string of characters that lets a program, such as a notebook, act for your account. A plan is the kind of access an account has, and the Open Plan is the one you do not pay for. An instance is your share of access under a plan.

IBM may ask for a credit card when you create the account. The card is not charged unless you create a Pay-As-You-Go instance and use it to send work to a quantum computer. If you cannot or do not want to give one, ask an organizer on Discord. Challenges 3 and 4 need no account.

In the Accounts section of the Session 1 notebook the steps are cells you can run: IBM Quantum Platform first, then the second option. A job is one request to run circuits, and a processor is the chip that holds the qubits. Jobs on IBM processors always go through your own IBM Quantum account. The second option does not replace that account. It adds the other quantum computers that qBraid offers.

In this notebook "the first cell" means the three code cells of the Setup section above. If they ran without an error, you are ready.

## Part 1: The seminar ideas

This part follows tonight's slides in order. Each heading is the title of a slide. Under it you find a short recap, the math from the slide, and often a small cell to run. The sections called "Derivation" show the steps behind a result, one at a time.

**If you missed the earlier sessions.** Tonight builds on a few ideas from Monday. Here they are, each in a sentence or two.

- **Qubit.** The quantum version of a bit. A bit is the unit of an ordinary computer, something that is either 0 or 1. When you measure a qubit, you read a 0 or a 1.
- **Bitstring.** What you read when you measure several qubits: a string of 0s and 1s with one character per qubit, such as `011`. Qiskit numbers the qubits from 0 and prints qubit 0 as the right-most character.
- **Amplitude and state vector.** Before the measurement, the state of $n$ qubits is described by one number for each of the $2^n$ possible bitstrings. These numbers are the amplitudes, and the list of all of them is the state vector. An amplitude is in general a complex number, a number of the form $x + iy$ with $i^2 = -1$.
- **Probability.** The probability of reading a bitstring is the squared size of its amplitude. The size of $x + iy$ is $\sqrt{x^2 + y^2}$, and the size of an ordinary number is the number without its sign. All the probabilities add up to 1.
- **Ket and superposition.** $\lvert 011 \rangle$ is called a ket. It is the state in which a measurement is certain to give 011. A general state is a sum of kets, each multiplied by its amplitude. A state with more than one amplitude different from zero is a superposition.
- **Gate.** An operation that changes the amplitudes without measuring anything. Tonight's gates: H, the Hadamard gate, turns $\lvert 0 \rangle$ into the equal superposition $\frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$. CX acts on two qubits, a control and a target, and flips the target when the control is 1. A rotation gate turns the state of one qubit by an angle that you choose.
- **Circuit.** A list of gates applied to qubits, usually ending with a measurement of every qubit.
- **Entanglement.** Several qubits are entangled when they share one state that cannot be described one qubit at a time.
- **GHZ state.** The entangled state $\frac{\lvert 00 \ldots 0 \rangle + \lvert 11 \ldots 1 \rangle}{\sqrt{2}}$ of $n$ qubits: an equal superposition of "every qubit is 0" and "every qubit is 1". The letters are the initials of Greenberger, Horne and Zeilinger, the physicists it is named after. One H gate and a chain of $n - 1$ CX gates build it.
- **Shots and counts.** One run of a circuit is called a shot, and it gives one bitstring. The counts say how many shots gave each bitstring.

The cell below shows these ideas on three qubits. It builds a GHZ state gate by gate and prints, after each gate, the amplitudes that are not zero. Then it measures the state 1,000 times with a sampler, the Qiskit tool that runs a circuit for a number of shots and returns the measured bitstrings.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector


def nonzero_amplitudes(circuit):
    """The amplitudes of a circuit that are not zero, with the labels Qiskit prints.

    Only the real part is kept. That is the whole amplitude for circuits made of H and CX gates.
    """
    width = circuit.num_qubits
    values = Statevector(circuit).data            # the state vector: a list of 2**width complex numbers
    shown = {}
    for k in range(2 ** width):
        if abs(values[k]) > 1e-9:
            shown[format(k, f"0{width}b")] = round(float(values[k].real), 3)
    return shown


warm_up = QuantumCircuit(3)        # a circuit with three qubits, all starting in 0
print("start:                  ", nonzero_amplitudes(warm_up))
warm_up.h(0)                       # H on qubit 0
print("after H on qubit 0:     ", nonzero_amplitudes(warm_up))
warm_up.cx(0, 1)                   # CX with control qubit 0 and target qubit 1
print("after CX on qubits 0, 1:", nonzero_amplitudes(warm_up))
warm_up.cx(1, 2)                   # CX with control qubit 1 and target qubit 2
print("after CX on qubits 1, 2:", nonzero_amplitudes(warm_up))

# Measure every qubit, 1,000 shots. A sampler runs a circuit and returns the measured bitstrings.
measured = warm_up.measure_all(inplace=False)
seeded_sampler = StatevectorSampler(seed=np.random.default_rng(2026))     # the seed fixes the random numbers
print("counts in 1,000 shots:  ", seeded_sampler.run([measured], shots=1000).result()[0].data.meas.get_counts())

Read the output line by line.

- At the start the only amplitude is on `000`: all three qubits are 0.
- After H there are two equal amplitudes, on `000` and `001`. The character that changed is the right one, because qubit 0 is printed on the right. 0.707 is $\frac{1}{\sqrt{2}}$ rounded to three decimals, so each of the two results has the probability $\left( \frac{1}{\sqrt{2}} \right)^2 = 0.5$.
- Each CX then flips one more qubit in the part of the state where its control is 1. `001` becomes `011` and then `111`. The part `000` never changes.
- The final state is the GHZ state on three qubits. In 1,000 shots, `000` came up 464 times and `111` came up 536 times. No other bitstring appeared.

Three qubits have $2^3 = 8$ possible bitstrings, so their state vector has eight amplitudes. Here six of them are zero. Keep both facts in mind: the number of amplitudes doubles with every qubit, and in some states most of them carry no information. The first half of tonight is about these two facts.

### Simulating $n$ qubits takes $2^n$ complex numbers

*Slide 7.*

$$\text{memory} = 2^{n} \times 16 \; \text{bytes}$$

30 qubits: 16 GiB. 40 qubits: 16 TiB. 50 qubits: 16 PiB.
Every added qubit doubles it.

A simulator has to keep track of the state of the qubits. The most direct way is to store the whole state vector, one amplitude for each of the $2^n$ bitstrings. A simulator that works this way is called a state-vector simulator. It is exact: nothing is approximated. One amplitude is a complex number, and a computer stores it in 16 bytes. A byte is the basic unit of computer memory. So the state of $n$ qubits takes $2^n \times 16$ bytes.

The units on the slide: a gibibyte (GiB) is $2^{30}$ bytes, a little more than a billion bytes, and 16 GiB is the memory of a well-equipped laptop. A tebibyte (TiB) is 1,024 GiB, and a pebibyte (PiB) is 1,024 TiB. Ten more qubits multiply the memory by $2^{10} = 1{,}024$, which is one step up this ladder each time: 16 GiB at 30 qubits, 16 TiB at 40, 16 PiB at 50.

A quantum computer does not store these numbers anywhere. Its qubits are the state. A simulator has to write the state down, and past some number of qubits no computer has the memory for it. Tonight's slides call that limit the wall.

The cell below asks Qiskit for the state vector of $n$ qubits that are all 0, for three values of $n$. It prints how many bytes the state vector really takes in the computer's memory, next to what the formula gives. The computer in question, here and in the lab, is the one that runs this notebook: your laptop, or the Colab or qBraid Lab server.

In [ ]:
from qiskit.quantum_info import Statevector

for n_qubits in (10, 20, 24):
    amplitudes = Statevector.from_label("0" * n_qubits).data      # the state vector of n qubits that are all 0
    print(f"{n_qubits} qubits: {len(amplitudes):,} amplitudes, {amplitudes.nbytes:,} bytes in memory."
          f" The formula gives {16 * 2 ** n_qubits:,}.")
del amplitudes      # hand the memory of the last state vector back

The two numbers agree in every line. The state vector of 24 qubits takes 268,435,456 bytes, which is 256 MiB (a mebibyte, MiB, is $2^{20}$ bytes). Six more qubits multiply that by $2^6 = 64$ and give the 16 GiB of the slide.

#### Derivation: Counting the bytes

*Slide 8.*

Amplitudes in the state. Each of the $n$ qubits reads 0 or 1, so there are $2 \times 2 \times \cdots \times 2$ possible bitstrings, with one factor of 2 for each qubit. The state has one amplitude for each bitstring.

$$2^n$$

Bytes per amplitude: two doubles. An amplitude is a complex number $x + iy$, so the computer stores two ordinary numbers, $x$ and $y$. Each is stored as a double, short for double-precision number. That is the standard format for a number with decimals. It takes 8 bytes and holds about 16 digits.

$$2 \times 8 = 16$$

30 qubits. Write 16 as $2^4$. Powers of two multiply by adding their exponents. One GiB is $2^{30}$ bytes, so $2^{34}$ bytes are $2^4 = 16$ GiB.

$$2^{30} \times 2^4 = 2^{34} \text{ bytes} = 16 \text{ GiB}$$

Each added qubit. Going from $n$ qubits to $n + 1$ doubles the number of amplitudes, and with it the memory.

$$\frac{2^{n+1}}{2^n} = 2$$

Ten added qubits double the memory ten times, which is a factor of $2^{10} = 1{,}024$. This is why laptops stop near 30 qubits and the largest clusters stop near 50. The record, reported in 2025 from the JUPITER supercomputer in Jülich, Germany, is a full simulation of 50 qubits (H. De Raedt et al., arXiv:2511.03359). It stored each amplitude in fewer bytes than the 16 counted here; at 16 bytes, 50 qubits would need 16 PiB. A cluster is a large computer made of many computers that work together, and you meet one below, under "Palmetto 2 runs the simulations a laptop cannot".

### GPUs speed up the same linear algebra

*Slide 9.*

- Applying a gate is one large, regular matrix operation over all amplitudes.
- That is the kind of work a GPU does well.

Here is what a state-vector simulator does when a gate acts on one qubit. It goes through all $2^n$ amplitudes in pairs. The two amplitudes of a pair belong to two bitstrings that differ only in that qubit's bit. Each pair is replaced by two new numbers, computed from the old pair with the four numbers of the gate's matrix. A gate on one qubit is described by a 2 by 2 matrix, a table of four numbers that say how much of each old amplitude goes into each of the two new ones. The same small calculation is repeated $2^{n-1}$ times, and no pair has to wait for another.

A GPU, a graphics processing unit, is a chip that was first built for drawing images, where the same small calculation is repeated for every point of the screen. It has thousands of simple arithmetic units that work side by side. A long list of identical, independent calculations is the work it does well, so a GPU can speed up a state-vector simulation.

A GPU changes the speed. It does not change the count from the slides before: the state vector still has $2^n$ amplitudes, and they have to fit in the GPU's own memory.

This notebook has no GPU cell, and nothing in it needs a GPU. To run a simulation on a GPU you need a computer that has a suitable one and a simulator that was built to use it. The GPU nodes of Palmetto 2, the cluster of the next slide, are computers of that kind.

### Palmetto 2 runs the simulations a laptop cannot

*Slide 10.*

- **What it is.** Clemson's high-performance computing cluster: more than a thousand compute nodes, many with GPUs.
- **How you get on.** No cost for Clemson students, faculty, and staff. A faculty member sponsors your account.
- **How it runs.** You submit a job to the Slurm scheduler, and it runs when the nodes you asked for are free.

The words on the cards. A compute node is one of the computers that make up a cluster. The nodes are joined by a fast network, so that a single program can use many of them at once. A job is a request to run a program, together with a list of what it needs: how many nodes, how much memory, how much time. You do not start a program on a cluster yourself. You hand the job to the scheduler, the program that decides which job runs on which nodes and when. On Palmetto 2 the scheduler is called Slurm. The second card means that a student does not sign up alone: a faculty member has to sponsor the account.

What a cluster buys you, in qubits. More nodes mean more memory, but every doubling of the memory adds only one qubit to a state vector. A thousand times the memory adds ten qubits, because $2^{10} = 1{,}024$. A cluster moves the memory limit from near 30 qubits, where a laptop stops, into the 40s. It does not remove the wall.

### Tensor networks store entanglement, not amplitudes

*Slide 11.*

- A matrix product state keeps one small tensor per qubit.
- Neighbors are joined by bonds of size $\chi$.
- Memory grows like $n\chi^2$, not $2^n$.

A state vector writes down every amplitude, even when most of them are zero or follow a simple pattern. The GHZ state at the top of this part is the extreme case: $2^n$ amplitudes, and all but two of them are zero. A tensor network is a way to store a state in a compressed form that uses such patterns.

The words on the slide. A tensor is a table of numbers. A number in the table is found by its indices, the positions that say where it sits. A list of numbers has one index. A matrix, a table with rows and columns, has two: the row and the column. A matrix product state is the simplest tensor network: a chain with one small tensor for each qubit. Each tensor is joined to its two neighbors in the chain by a bond, which is an index that the two neighbors share. The size of a bond, written $\chi$ (the Greek letter chi), is the number of values that this shared index can take. The larger $\chi$ is, the more the two sides of the bond can share.

The numbers behind the third bullet. Each tensor has $\chi \times 2 \times \chi$ entries: $\chi$ for the bond on its left, 2 for the two values of its own qubit, and $\chi$ for the bond on its right. All $n$ tensors together hold about $2n\chi^2$ numbers. For $n = 100$ and $\chi = 64$ that is 819,200 complex numbers, about 13 MB (a megabyte, MB, is a million bytes). The full state vector of 100 qubits could not be stored anywhere.

The price is in the title. The chain does not store amplitudes. It stores what neighboring parts of the chain share with each other, which is their entanglement, and a bond of size $\chi$ can carry only a limited amount of it. A GHZ state needs only $\chi = 2$. A generic, highly entangled state needs $\chi = 2^{n/2}$, and then the saving disappears. The derivation below makes both statements precise.

The picture on the slide shows the state vector as one long row of cells and, under it, a chain of six boxes, one per qubit, joined by bonds labeled $\chi$. The cell below draws the same chain. It numbers the qubits from 1, as the derivation below does, while the slide labels them q0 to q5.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 2.2))
for k in range(6):
    # One box for the tensor of each qubit.
    ax.add_patch(plt.Rectangle((k - 0.3, -0.3), 0.6, 0.6, facecolor="tab:blue", edgecolor="black", zorder=2))
    ax.text(k, 0, f"$A^{{s_{k + 1}}}$", ha="center", va="center", color="white", fontsize=11, zorder=3)
    # The line under a box stands for the qubit's own bit, 0 or 1.
    ax.plot([k, k], [-0.3, -0.85], color="black", lw=2, zorder=1)
    ax.text(k, -1.08, f"qubit {k + 1}", ha="center", va="center", fontsize=9)
    # The orange line is the bond to the next tensor.
    if k < 5:
        ax.plot([k + 0.3, k + 0.7], [0, 0], color="tab:orange", lw=4, zorder=1)
        ax.text(k + 0.5, 0.22, r"$\chi$", ha="center", va="center", fontsize=11)
ax.set_xlim(-0.6, 5.6)
ax.set_ylim(-1.3, 0.6)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("A matrix product state: one small tensor per qubit")
plt.show()

Each blue box is the tensor of one qubit. The black line under a box stands for that qubit's own bit, $s_1$ for the first qubit, $s_2$ for the second, and so on. Here, as in the formulas below, the qubits are counted from 1: qubit 1 is the one Qiskit calls qubit 0, qubit 2 is Qiskit's qubit 1, and so on. The orange lines are the bonds. The labels $A^{s_1}$ to $A^{s_6}$ are the names the derivation below uses.

#### Derivation: What a matrix product state stores

*Slide 12.*

This derivation writes the first qubit as the left character of a bitstring, as the slides do. Qiskit prints qubit 0 as the right character. For the GHZ state used in the examples both orders read the same.

The full state, written $\lvert \psi \rangle$ with the Greek letter psi. Here $s_1$ is the bit of the first qubit, $s_2$ the bit of the second, and so on. The sum runs over all $2^n$ bitstrings, and $c_{s_1 \ldots s_n}$ is the amplitude of the bitstring $s_1 \ldots s_n$. This is the state vector, written as a formula.

$$\lvert \psi \rangle = \sum_{s_1 \ldots s_n} c_{s_1 \ldots s_n} \lvert s_1 \ldots s_n \rangle$$

Each amplitude is a product of small matrices. Every qubit has two matrices of its own, one for the bit value 0 and one for the bit value 1. The superscript says which of the two to take: $A^{s_1}$ is the first qubit's matrix for its bit $s_1$. The slide uses the same letter $A$ for every qubit, but the matrices of different qubits are in general different. To get the amplitude of one bitstring, pick for every qubit the matrix that goes with its bit and multiply the $n$ matrices in order.

$$c_{s_1 \ldots s_n} = A^{s_1} A^{s_2} \cdots A^{s_n}$$

One detail the slide leaves out. A product of matrices is a matrix, and an amplitude is a single number. The two ends of the chain settle this: the first qubit's $A^{s_1}$ is a single row and the last qubit's $A^{s_n}$ is a single column. A row times matrices times a column is one number.

Numbers stored, with an example. Each matrix has $\chi$ rows and $\chi$ columns, so it holds $\chi^2$ numbers. Every qubit has two matrices and there are $n$ qubits, which gives $2n\chi^2$ numbers. (The exact count is a little lower, because the two end tensors are smaller.)

$$2n\chi^2: \quad 2 \times 100 \times 64^2 = 819{,}200$$

Compare that with the state vector of 100 qubits, which has $2^{100} \approx 1.3 \times 10^{30}$ amplitudes.

The price: capped entanglement. Cut the chain between two neighboring qubits. $S$ is the entanglement entropy across that cut, a number that says how much entanglement the qubits on the left share with the qubits on the right. It is counted in bits. $S = 0$ means that the two sides are not entangled at all, and a GHZ state has $S = 1$ across every cut. A matrix product state with bonds of size $\chi$ cannot hold more than $\log_2 \chi$ bits across any cut. Here $\log_2$ is the logarithm to base 2: $\log_2 \chi = m$ means $2^m = \chi$.

$$S \leq \log_2 \chi$$

So a GHZ state, with $S = 1$, needs at least $\chi = 2$, because $\log_2 2 = 1$. The cell below shows that $\chi = 2$ is also enough. A generic, highly entangled state of $n$ qubits has close to $\frac{n}{2}$ bits across its middle cut and needs $\chi = 2^{n/2}$. With bonds of that size everywhere the formula gives $2n\chi^2 = 2n \times 2^n$. The bonds near the two ends of the chain can be smaller, and the count that is really needed is between two and three times the $2^n$ amplitudes of the state vector itself. Either way the saving has disappeared.

The cell below writes the GHZ state on 60 qubits as a matrix product state with $\chi = 2$. It stores two rows for the first qubit, two 2 by 2 matrices for every qubit in the middle, and two columns for the last qubit. The function `amplitude` multiplies one of them per qubit, as in the second step above. In Python, `@` multiplies matrices.

In [ ]:
import numpy as np

# The GHZ state as a matrix product state with bonds of size 2.
# First qubit: a row for each bit. Middle qubits: a matrix for each bit. Last qubit: a column for each bit.
first = {"0": np.array([1, 0]) / np.sqrt(2), "1": np.array([0, 1]) / np.sqrt(2)}
middle = {"0": np.array([[1, 0], [0, 0]]), "1": np.array([[0, 0], [0, 1]])}
last = {"0": np.array([1, 0]), "1": np.array([0, 1])}


def amplitude(bits):
    """Multiply one small matrix per qubit, chosen by that qubit's bit."""
    product = first[bits[0]]
    for bit in bits[1:-1]:
        product = product @ middle[bit]
    return float(product @ last[bits[-1]])


size = 60
print("amplitude of 60 zeros:             ", round(amplitude("0" * size), 4))
print("amplitude of 60 ones:              ", round(amplitude("1" * size), 4))
print("amplitude of 30 zeros, then 30 ones:", round(amplitude("0" * 30 + "1" * 30), 4))
numbers_stored = 2 * 2 + (size - 2) * 2 * 4 + 2 * 2
print("numbers stored:", numbers_stored, "| the formula 2 n chi^2 gives", 2 * size * 2 ** 2)
print("amplitudes in the state vector of 60 qubits:", f"{2 ** size:,}")

The two amplitudes of the GHZ state come out as 0.7071, which is $\frac{1}{\sqrt{2}}$, and a bitstring that mixes zeros and ones gets the amplitude 0. The whole 60-qubit state is held in 472 numbers, a little under the 480 of the formula. Its state vector would have 1,152,921,504,606,846,976 amplitudes, more than a billion times a billion.

How do you find out how large a bond has to be for a given state? Cut the qubits into two groups. Every state can be written as a sum of terms, where each term is a state of the first group times a state of the second group. If one term is enough, the two groups are not entangled. The more terms it takes, the more entanglement crosses the cut. Linear algebra is the mathematics of lists and tables of numbers. One of its standard tools, the singular value decomposition, finds the shortest sum of this kind. It returns one number for each term, called a singular value, and the square of a singular value is the weight of its term. The number of singular values that are not zero is the bond size $\chi$ that the cut needs. The entanglement entropy follows from the same numbers. If $\sigma_1, \sigma_2, \ldots$ are the singular values, their squares add up to 1 and

$$S = -\sum_i \sigma_i^2 \log_2 \sigma_i^2$$

The next cell does this for the cut through the middle of 10 qubits. It arranges the 1,024 amplitudes as a table with 32 rows and 32 columns, where the row says what qubits 5 to 9 read and the column says what qubits 0 to 4 read, and hands the table to the singular value decomposition. It compares the GHZ state with a random state, one whose 1,024 amplitudes were drawn at random.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, random_statevector

ghz_10 = QuantumCircuit(10)        # the GHZ circuit on 10 qubits, without measurements
ghz_10.h(0)
for i in range(9):
    ghz_10.cx(i, i + 1)

for name, psi in (("GHZ state:   ", Statevector(ghz_10)), ("random state:", random_statevector(2 ** 10, seed=1))):
    table = psi.data.reshape(32, 32)                           # rows: qubits 5 to 9, columns: qubits 0 to 4
    singular_values = np.linalg.svd(table, compute_uv=False)
    kept = singular_values[singular_values > 1e-10]            # the ones that are not zero
    entropy = -np.sum(kept ** 2 * np.log2(kept ** 2))
    print(name, len(kept), "singular values are not zero | the largest three:", np.round(singular_values[:3], 4),
          "| entropy S =", round(float(entropy), 2), "bits")

The GHZ state has 2 singular values that are not zero, both 0.7071. A bond of size 2 is enough across this cut, and the entropy is 1 bit, exactly $\log_2 2$. The random state has all 32, the most a table with 32 rows can have, and an entropy of 4.28 bits, close to the largest possible value of $\log_2 32 = 5$. A matrix product state saves nothing for it.

That is the rule for tensor networks: cheap when entanglement is low, no help when it is high. Lab 1 lets you measure the cheap side of it with Qiskit's simulator, which has a state-vector method and a matrix-product-state method.

### Quantum-centric supercomputing: each machine does what it is good at

*Slide 15.*

- The quantum processor produces samples.
- The classical cluster does the heavy pre- and post-processing.
- The result sets up the next quantum run.

A quantum processor is the chip that holds the qubits of a real quantum computer. A sample is one measured bitstring, the result of one shot. Pre-processing is the work done before a circuit runs, such as building it and rewriting it for the processor. Post-processing is the work done on the samples afterwards to turn them into an answer. Both can be heavy computations in their own right, and they are work that an ordinary cluster does well.

The idea is a division of labor. The quantum processor is one part of a larger computation, an accelerator, in the way a GPU is: a special device that takes over the one step it is good at. The picture on the slide is a loop of three boxes: quantum processor, classical cluster, result. The loop closes because the result of one round decides which circuits the quantum processor runs next. At Clemson, the classical cluster in this picture would be Palmetto 2.

Tuesday's session (Session 2, whose notebook is on the event page) had a small loop of this kind: an ordinary program chose the angles of a circuit, the circuit ran, and the result went back to the program, which chose the next angles. Tonight's variational classifier, further down, is trained in the same way.

**The second half: machine learning.** The slides from here on are about classification, one of the standard tasks of machine learning. The words you need:

- **Data point and feature.** A data point is one example, described by a few numbers. Each of these numbers is called a feature. Tonight every data point has two features, so it is a pair of numbers $x = (x_1, x_2)$, and you can draw it as a point in a plane.
- **Label.** The group that a data point belongs to. Tonight there are two groups, with the labels 0 and 1.
- **Classifier.** A program that takes the features of a data point and predicts its label.
- **Training set and test set.** You show the classifier some data points together with their labels. These points are the training set, and adjusting the classifier to them is called training. Afterwards you try the classifier on other points, whose labels it has never seen. These are the test set.
- **Accuracy.** The fraction of points that the classifier labels correctly. The accuracy on the test set is the honest score, because a program can memorize the points it was trained on.

The cell below makes tonight's data set with scikit-learn, the Python library for classical machine learning that tonight's lab uses. `make_moons` draws 100 points in the shape of two interleaved half-moons, one half-moon for each label, with some random scatter. `train_test_split` puts 40 of the points in the training set and 60 in the test set, with equal numbers of both labels in each. The two lines after `train_test_split` rescale the features so that, in the training set, each feature runs from 0 to $\frac{\pi}{2}$. The reason is that tonight's circuits use the features as rotation angles. The scale is set from the training points alone, as it would have to be if the test points were not known yet.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X, y = make_moons(n_samples=100, noise=0.15, random_state=2026)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=40, stratify=y, random_state=2026)
scaler = MinMaxScaler(feature_range=(0, np.pi / 2)).fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

print("training set:", X_train.shape[0], "points with", X_train.shape[1], "features each")
print("test set:", X_test.shape[0], "points")
print("points with label 0 and with label 1: training set", np.bincount(y_train), "| test set", np.bincount(y_test))
print("first training point:", np.round(X_train[0], 3), "| its label:", y_train[0])

`X_train` is a table with 40 rows and 2 columns: one row for each training point, one column for each feature. `y_train` holds the 40 labels. `X_test` and `y_test` are the same for the 60 test points. The first training point has the features 0.599 and 1.246 and the label 0.

The next cell draws all 100 points. Filled circles are training points and open circles are test points.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.8))
for label, color in ((0, "tab:blue"), (1, "tab:orange")):
    train_points = X_train[y_train == label]          # the training points with this label
    test_points = X_test[y_test == label]             # the test points with this label
    ax.scatter(train_points[:, 0], train_points[:, 1], color=color, edgecolor="black", s=45,
               label=f"label {label}, training")
    ax.scatter(test_points[:, 0], test_points[:, 1], facecolor="none", edgecolor=color, s=45,
               label=f"label {label}, test")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("Tonight's data set: 40 training points, 60 test points")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.show()

The two half-moons curve around each other, so no straight line separates the blue points from the orange ones. That is what makes this small data set a fair test for a classifier. A few test points lie slightly outside the range from 0 to $\frac{\pi}{2}$, because the scale was set by the training points.

### A feature map turns a data point into a quantum state

*Slide 19.*

$$x \rightarrow \lvert \varphi(x) \rangle = U(x)\lvert 0 \ldots 0 \rangle$$

- Each feature sets a rotation angle.
- Entangling gates mix the features.
- This is the only place the data enters the circuit.

A quantum circuit cannot read a table of numbers. To use a data point in a circuit, you turn its features into the angles of gates. A feature map is a circuit $U(x)$ whose gates depend on the data point $x$. Every qubit starts in 0, which is written $\lvert 0 \ldots 0 \rangle$, the circuit runs, and the state at the end is called the feature state of $x$. It is written $\lvert \varphi(x) \rangle$, where $\varphi$ is the Greek letter phi. Different data points usually give different states.

The smallest example has one feature and one qubit. Take a single rotation gate, $R_y$, and use the feature as its angle.

$$\lvert \varphi(x) \rangle = R_y(x)\lvert 0 \rangle = \cos\frac{x}{2}\lvert 0 \rangle + \sin\frac{x}{2}\lvert 1 \rangle$$

The feature $x = 0$ gives $\lvert 0 \rangle$, the feature $x = \pi$ gives $\lvert 1 \rangle$, and every value in between gives a superposition of the two.

The picture on the slide is Qiskit's ZZ feature map for two features, with its block of gates once. It uses one qubit for each feature. The cell below draws it.

In [ ]:
from qiskit.circuit.library import zz_feature_map

demo_map = zz_feature_map(2, reps=1)      # two features, and the basic block of gates once
demo_map.draw("mpl")

Read the drawing from left to right. `x[0]` and `x[1]` are the two features, $x_1$ and $x_2$ in the text (Python counts from 0).

- An H gate on each qubit makes an equal superposition of all four bitstrings.
- Each P gate uses one feature. P is the phase gate, a rotation gate: it leaves the amplitude of $\lvert 0 \rangle$ alone and turns the phase of the amplitude of $\lvert 1 \rangle$ by the angle written in its box. The phase of a complex number is its direction in the plane of complex numbers. For an ordinary number it is the sign. This is the first bullet of the slide: each feature sets a rotation angle, here $2x_1$ on qubit 0 and $2x_2$ on qubit 1.
- The last three gates, CX, P, CX, act on both qubits together. They turn the phase of the amplitudes of `01` and `10`, the two bitstrings in which the qubits differ, by an angle that contains both features: $2(\pi - x_1)(\pi - x_2)$. The box in the drawing writes the same angle as `(-π + x[0])*(-π + x[1])*2`, where the two minus signs cancel. This is the second bullet: entangling gates mix the features. An entangling gate is a gate on two qubits that can entangle them, such as CX. The letters ZZ in the name refer to this block.

The third bullet says that the features appear in these gate angles and nowhere else. Everything that is done afterwards is done to the feature state. If the feature map gives two different data points the same state, nothing later in the circuit can tell them apart again. So the choice of the feature map, and of how the features are scaled into angles, decides a lot. Lab 3 comes back to both.

`reps=1` asks for this block of gates once. The lab uses `reps=2`, which is the whole drawing twice in a row.

The next cell puts the first training point into the feature map and prints its feature state.

In [ ]:
from qiskit.quantum_info import Statevector

point = X_train[0]                                    # one data point: two features
bound_map = demo_map.assign_parameters(point)         # the feature map with the two numbers filled in
feature_state = Statevector(bound_map)                # the state at the end of that circuit
print("data point:", np.round(point, 3))
print("its feature state, four amplitudes:", np.round(feature_state.data, 3))
print("sizes of the four amplitudes:", np.round(np.abs(feature_state.data), 3))
print("probabilities of 00, 01, 10, 11:", np.round(feature_state.probabilities(), 3))

The data point has become a state of two qubits with four complex amplitudes. They are listed in the order `00`, `01`, `10`, `11`, with qubit 0 as the right character. In Python `j` is the number $i$. All four amplitudes have the size 0.5, so a measurement of this state gives each of the four bitstrings with probability 0.25, for this data point and for every other one. With one repetition of the block, the data sits only in the phases of the amplitudes, and a direct measurement cannot see a phase. The kernel of the next slide is a way to get at it.

### A quantum kernel is an overlap you can measure

*Slide 20.*

$$k\left( x, x' \right) = \left\lvert \langle \varphi(x) \vert \varphi(x') \rangle \right\rvert^{2}$$

Run $U(x)$, then the inverse of $U(x')$, and count how often every qubit reads 0.

A kernel is a function that takes two data points and returns one number that says how alike they are. Here $x$ and $x'$ (read "x prime") are two data points. The symbol $\langle \varphi(x) \vert \varphi(x') \rangle$ is the overlap of their two feature states. To compute it, write both states as lists of amplitudes, replace every amplitude in the first list by its complex conjugate (the conjugate of $a + ib$ is $a - ib$), multiply the two lists entry by entry, and add up the products. The quantum kernel is the squared size of that number. It is 1 when the two feature states are the same. It is 0 when they are as different as two states can be, which means that a single measurement could tell them apart with certainty.

For the one-qubit feature map from the section above, the overlap can be worked out by hand.

$$\langle \varphi(x) \vert \varphi(x') \rangle = \cos\frac{x}{2}\cos\frac{x'}{2} + \sin\frac{x}{2}\sin\frac{x'}{2} = \cos\frac{x - x'}{2}, \qquad k\left( x, x' \right) = \cos^2\frac{x - x'}{2}$$

This kernel is 1 for two equal features. It falls as the features move apart and reaches 0 when they differ by $\pi$.

Why this is useful. A support vector machine is a classical machine-learning method. When it is used for classification it is called a support vector classifier. It never looks at the features themselves. All it needs is the kernel value of every pair of points. During training it gives every training point a weight, and afterwards it labels a new point by comparing it with the training points: each training point votes for its own label, and its vote counts more when its weight and its kernel value with the new point are larger. So the work can be divided. A quantum circuit supplies the kernel values, and the classical method does everything after that.

The second line of the slide says how to get a kernel value out of a quantum computer, which cannot print its amplitudes. The first derivation below shows why that recipe works. The second one counts what it costs.

The cell below checks the formula for the one-qubit map. `a.inner(b)` computes the overlap of two state vectors, with the complex conjugate taken care of.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def toy_state(x):
    """The feature state of the one-qubit feature map R_y(x)."""
    toy = QuantumCircuit(1)
    toy.ry(x, 0)
    return Statevector(toy)


for x_a, x_b in ((0.4, 0.4), (0.4, 1.0), (0.4, 0.4 + np.pi)):
    overlap = toy_state(x_a).inner(toy_state(x_b))
    print(f"x = {x_a:.2f}, x' = {x_b:.2f}: kernel from the two states {abs(overlap) ** 2:.4f}"
          f" | from the formula {np.cos((x_a - x_b) / 2) ** 2:.4f}")

The two columns agree. A point with itself gives 1. Two features that differ by 0.6 give 0.9127. Two features that differ by $\pi$ give 0.

#### Derivation: Why the all-zeros count is the kernel

*Slide 21.*

In this derivation the ket $\lvert 0 \rangle$ stands for every qubit in 0.

Feature state. The feature map, applied to qubits that all start in 0, gives the feature state of $x$.

$$\lvert \varphi(x) \rangle = U(x)\lvert 0 \rangle$$

Run one map, then undo the other. $U^{\dagger}$ (read "U dagger") is the inverse of the circuit $U$: the same gates in reverse order, each one undone. Running $U$ and then $U^{\dagger}$ for the same data point brings every state back to where it started. Here the two data points are different: first the map for $x$, then the inverse of the map for $x'$.

$$U^{\dagger}(x')\,U(x)\lvert 0 \rangle$$

Amplitude on all zeros. An overlap is written in two halves. The right half is a ket. The left half is called a bra, such as $\langle 0 \rvert$, and it stands for the same state with every amplitude replaced by its complex conjugate. The amplitude of the all-zeros result in a state is the overlap of $\lvert 0 \rangle$ with that state, so put the bra $\langle 0 \rvert$ in front of the state from the step before. Now group the symbols differently. The ket of the second data point is $U(x')\lvert 0 \rangle$, and the bra that belongs to it is $\langle 0 \rvert U^{\dagger}(x')$: turning a ket into a bra reverses the order and turns $U$ into $U^{\dagger}$. So the left half of the expression is the bra $\langle \varphi(x') \rvert$ and the right half is the ket $\lvert \varphi(x) \rangle$.

$$\langle 0 \rvert U^{\dagger}(x')\,U(x) \lvert 0 \rangle = \langle \varphi(x') \vert \varphi(x) \rangle$$

Probability of all zeros. The probability of a result is the squared size of its amplitude. Swapping the two data points turns the overlap into its complex conjugate, which has the same size, so the result is the kernel of the slide.

$$P(0 \ldots 0) = \left\lvert \langle \varphi(x') \vert \varphi(x) \rangle \right\rvert^2 = k\left( x, x' \right)$$

So the fraction of shots in which every qubit reads 0 is an estimate of the kernel. This circuit is called compute and uncompute, and it is how a kernel is measured on hardware, which means on a real quantum computer. On a simulator the same number can be read straight from the two state vectors, which is what tonight's lab does.

The cell below builds the compute-and-uncompute circuit for two made-up data points, with the feature map `demo_map` from above. `a.compose(b)` makes the circuit that runs `a` and then `b`, and `b.inverse()` is the circuit that undoes `b`. The cell prints the kernel from the two state vectors and the exact probability of `00` at the end of the circuit, then draws the circuit.

In [ ]:
x_a, x_b = [0.3, 1.2], [0.9, 0.4]                     # two made-up data points

state_a = Statevector(demo_map.assign_parameters(x_a))
state_b = Statevector(demo_map.assign_parameters(x_b))
print("kernel from the two state vectors:", round(abs(state_a.inner(state_b)) ** 2, 4))

# U(x_a), then the inverse of U(x_b)
pair = demo_map.assign_parameters(x_a).compose(demo_map.assign_parameters(x_b).inverse())
print("probability of 00, exact:         ", round(float(Statevector(pair).probabilities()[0]), 4))
pair.draw("mpl")

Both numbers are 0.2731. In the drawing, the left half, up to the second CX, is the feature map for the first point. The right half is the feature map for the second point run backwards: the same kinds of gates in the opposite order, with the signs of the angles reversed.

A real quantum computer cannot print a probability. It can only run shots. The next cell measures both qubits at the end of the circuit and runs 4,096 shots.

In [ ]:
from qiskit.primitives import StatevectorSampler

measured_pair = pair.measure_all(inplace=False)                           # a copy of the circuit, with measurements
seeded_sampler = StatevectorSampler(seed=np.random.default_rng(13))       # the seed fixes the random numbers
pair_counts = seeded_sampler.run([measured_pair], shots=4096).result()[0].data.meas.get_counts()
print("counts in 4,096 shots:", pair_counts)
print("fraction of shots that read 00:", round(pair_counts.get("00", 0) / 4096, 4))

1,104 of the 4,096 shots read `00`, a fraction of 0.2695. The exact value is 0.2731. The difference of 0.0036 is shot noise, the random scatter of a count around its expected value. The next derivation says how many shots a given precision takes.

#### Derivation: What a kernel matrix costs

*Slide 22.*

Pairs among $N$ training points. A support vector classifier needs the kernel value of every pair of training points. Arrange these numbers in a table with $N$ rows and $N$ columns, the kernel matrix: the entry in row $i$ and column $j$ is the kernel of point $i$ with point $j$. The $N$ entries on the diagonal compare a point with itself. They equal 1 and need no circuit. The other $N(N-1)$ entries come in equal pairs, because the kernel of $i$ with $j$ is the kernel of $j$ with $i$. One circuit per pair is enough.

$$\frac{N(N-1)}{2}$$

For 40 points, the size of tonight's training set:

$$\frac{40 \times 39}{2} = 780 \text{ circuits}$$

Shots for precision $\delta$ on one entry. Every shot of the compute-and-uncompute circuit reads all zeros with probability $k$, like the toss of a coin that is not fair. The fraction of all-zero shots is an estimate of $k$. Its typical error, called the standard error, is

$$\sqrt{\frac{k(1-k)}{\text{shots}}}$$

Set this equal to the precision $\delta$ that you want and solve for the number of shots.

$$\text{shots} \approx \frac{k(1-k)}{\delta^2}$$

Worst case $k = \frac{1}{2}$, $\delta = 0.01$. The product $k(1-k)$ is largest at $k = \frac{1}{2}$, where it equals $\frac{1}{4}$.

$$\frac{1}{4 \times 0.01^2} = 2{,}500 \text{ shots}$$

Put the two counts together: 780 circuits with up to 2,500 shots each are close to two million shots for one kernel matrix of 40 points. The number of circuits grows like $N^2$, so a training set of twice the size needs about four times as many. This quadratic growth is the practical limit of kernel methods.

One more cost is not on the slide. The 780 circuits cover the training pairs only. To label a new point, the classifier needs the kernel of that point with every training point, which is one more circuit for each test point and each training point. For tonight's 60 test points that is $60 \times 40 = 2{,}400$ further circuits.

The cell below draws the number of shots against $k$ for $\delta = 0.01$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

delta = 0.01                                   # the precision you want on one kernel entry
k_values = np.linspace(0, 1, 101)              # 101 kernel values from 0 to 1
shots_needed = k_values * (1 - k_values) / delta ** 2
print("most shots needed:", round(float(shots_needed.max())), "at k =", k_values[shots_needed.argmax()])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(k_values, shots_needed, color="tab:blue")
ax.plot(0.5, 2500, "o", color="black")
ax.annotate("k = 0.5: 2,500 shots", (0.5, 2500), textcoords="offset points", xytext=(0, -16), ha="center")
ax.set_xlabel("kernel value k")
ax.set_ylabel("shots needed")
ax.set_title("Shots for a precision of 0.01 on one kernel entry")
ax.grid(True, alpha=0.3)
plt.show()

The curve peaks at 2,500 shots for $k = 0.5$. Kernel values near 0 or near 1 need fewer shots. The same formula explains the cell before: with 4,096 shots at $k = 0.2731$ the standard error is $\sqrt{0.2731 \times 0.7269 / 4096} \approx 0.007$, and the estimate there was off by 0.0036.

### A variational classifier trains the circuit itself

*Slide 23.*

$$f(x; \theta) = \langle 0 \rvert\, U^{\dagger}(x)\, W^{\dagger}(\theta)\, Z\, W(\theta)\, U(x) \lvert 0 \rangle$$

A second way: put the classifier into the circuit, as a block of gates whose angles are trained.

A positive $f$ predicts label 1, a negative one predicts label 0. $\theta$ is trained like the weights of a network.

A kernel method uses the quantum circuit only to compare data points. The classifier itself is classical. A variational classifier puts the classifier into the circuit as well. After the feature map $U(x)$ comes a second block of gates, $W(\theta)$. Its rotation angles are not data. They are free numbers, written together as $\theta = (\theta_1, \theta_2, \ldots)$, where $\theta$ is the Greek letter theta. At the end one qubit is measured.

How to read the formula. The state before the measurement is $W(\theta)\,U(x)\lvert 0 \rangle$: all qubits in 0, then the feature map, then the trainable block. The $Z$ in the middle stands for the measurement of one qubit, with the result 0 counted as $+1$ and the result 1 counted as $-1$. The whole expression is the average of these values over many shots. Such an average is called an expectation value. Here it equals the probability of reading 0 minus the probability of reading 1, so $f$ is a number between $-1$ and $+1$.

The classifier predicts one label when $f$ is positive and the other when $f$ is negative. To train it, you define a loss, a number that is small when the predictions on the training set are good. An optimizer, a classical program that searches for the smallest value of a function, then changes the angles $\theta$ step by step to lower the loss. A neural network, the standard model of machine learning, is trained in the same way, and its adjustable numbers are called weights. That is the comparison on the slide. The word variational means that the circuit has adjustable angles that are tuned in this way.

The cells below build a small variational classifier for tonight's data set and train it. The first cell makes the circuit. `efficient_su2` is a ready-made trainable block from Qiskit's library: rotation gates $R_y$ and $R_z$ on each qubit, with CX gates in between. An estimator is the Qiskit tool that returns expectation values, as a sampler returns bitstrings. `StatevectorEstimator` computes them exactly on your computer, without shots.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

trainable = efficient_su2(2, reps=2)        # W(theta): a block of gates with trainable angles
model = demo_map.compose(trainable)         # U(x) first, then W(theta)
z_qubit_0 = SparsePauliOp("IZ")             # what to measure: Z on qubit 0, I (nothing) on qubit 1. The right-most letter belongs to qubit 0.
estimator = StatevectorEstimator()
print("trainable angles:", trainable.num_parameters)
model.draw("mpl", fold=-1, scale=0.7)

The gates up to the second CX are the feature map $U(x)$ from above. The gates after it are $W(\theta)$, with 12 angles named `θ[0]` to `θ[11]`.

The next cell defines $f$ and tries it with 12 random angles. The labels 0 and 1 are rewritten as the targets $-1$ and $+1$, so that a correct prediction means that $f$ has the sign of the target.

In [ ]:
t_train = 2 * y_train - 1        # the labels 0 and 1, rewritten as the targets -1 and +1
t_test = 2 * y_test - 1


def f(points, theta):
    """The expectation value of Z on qubit 0 for every row of points, with the angles theta."""
    bound = model.assign_parameters(dict(zip(trainable.parameters, theta)))    # fill in the 12 angles
    job = estimator.run([(bound, z_qubit_0, points)])       # each row of points fills in x[0] and x[1]
    return job.result()[0].data.evs


def accuracy_of(points, targets, theta):
    """The fraction of points for which f has the sign of the target."""
    return float(np.mean(np.sign(f(points, theta)) == targets))


theta_start = np.random.default_rng(2026).uniform(-np.pi, np.pi, trainable.num_parameters)     # 12 random angles
print("f for the first three training points:", np.round(f(X_train[:3], theta_start), 3))
print("their targets:", t_train[:3])
print("accuracy with random angles: training set", accuracy_of(X_train, t_train, theta_start),
      "| test set", round(accuracy_of(X_test, t_test, theta_start), 3))

With random angles the classifier knows nothing. For the first three training points $f$ is 0.281, $-0.376$ and 0.47, while all three targets are $-1$, so only the second one has the right sign. The accuracy is 0.5 on the training set and 0.433 on the test set, which is what guessing gives.

Now the training. The loss is the mean squared error: the average, over the training points, of the squared distance between $f$ and the target. The optimizer is COBYLA, from SciPy, a Python library for scientific computing. It needs no slopes: it tries a set of angles, looks at the loss, and decides what to try next. `maxiter=80` stops it after 80 tries. The cell takes a few seconds.

In [ ]:
from scipy.optimize import minimize

history = []                     # the loss of every try, for the plot below


def loss(theta):
    """Mean squared distance between f and the targets on the training set."""
    value = float(np.mean((f(X_train, theta) - t_train) ** 2))
    value = round(value, 10)     # rounded, so that tiny rounding differences between computers do not change the path
    history.append(value)
    return value


trained = minimize(loss, theta_start, method="COBYLA", options={"maxiter": 80})
theta_trained = trained.x
print("loss with random angles:", round(history[0], 4), "| after training:", round(float(trained.fun), 4),
      "| tries:", len(history))
print("accuracy after training: training set", accuracy_of(X_train, t_train, theta_trained),
      "| test set", round(accuracy_of(X_test, t_test, theta_trained), 3))

The loss fell from 1.2033 to 0.3723. The trained circuit labels 39 of the 40 training points correctly, an accuracy of 0.975, and 54 of the 60 test points, an accuracy of 0.9. It has learned a useful rule from 40 examples.

Two remarks. An optimizer such as COBYLA reacts to the smallest differences in rounding between computers, which is why `loss` rounds its result. Its path can also change from one version of SciPy to the next. If your numbers differ a little from the ones above, one of these is the reason. And note what the training took: 80 tries, each of which evaluates the circuit for all 40 training points. On hardware every one of those 3,200 evaluations would need its own shots.

The next cell draws the loss of every try.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, len(history) + 1), history, color="tab:blue")
ax.set_xlabel("try number")
ax.set_ylabel("loss on the training set")
ax.set_title("Training the variational classifier")
ax.grid(True, alpha=0.3)
plt.show()

The loss jumps around during the first tries, while the optimizer changes one angle at a time to learn how the loss responds. Then it falls quickly, and after about 60 tries it hardly moves. More tries would not buy much: with `maxiter=300` the loss ends at 0.371.

The last cell of this section shows what the trained classifier has learned. It computes $f$ on a grid of points that covers the plane and draws it as colors, with the test points on top.

In [ ]:
side = np.linspace(-0.15, np.pi / 2 + 0.15, 40)
grid_1, grid_2 = np.meshgrid(side, side)                                # a grid of 40 by 40 points
grid_points = np.column_stack([grid_1.ravel(), grid_2.ravel()])         # one row for each grid point
grid_f = f(grid_points, theta_trained).reshape(grid_1.shape)            # f at every grid point

fig, ax = plt.subplots(figsize=(6.5, 3.8))
shade = ax.contourf(grid_1, grid_2, grid_f, levels=np.linspace(-1, 1, 9), cmap="RdBu_r", alpha=0.6)
ax.contour(grid_1, grid_2, grid_f, levels=[0], colors="black", linewidths=1)       # the line where f = 0
for label, color in ((0, "tab:blue"), (1, "tab:orange")):
    test_points = X_test[y_test == label]                               # the test points with this label
    ax.scatter(test_points[:, 0], test_points[:, 1], color=color, edgecolor="black", s=30,
               label=f"test point, label {label}")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("f after training")
fig.colorbar(shade, ax=ax, label="f")
ax.legend(fontsize=8, loc="upper right")
plt.show()

Red areas are where $f$ is positive, so the classifier predicts label 1 there. Blue areas are where $f$ is negative and the prediction is label 0. The black line is $f = 0$, the border between the two predictions. A blue dot on red ground, or an orange dot on blue ground, is a test point that the classifier gets wrong. With the numbers above there are six of them: the 60 test points minus the 54 that are right.

#### Derivation: The parameter-shift rule

*Slide 25.*

COBYLA searched without slopes. Most training methods, including the ones used for neural networks, need the gradient: the slope of $f$ with respect to each angle, which says in which direction to turn that angle. This derivation shows how a quantum computer can measure that slope. It follows one angle $\theta$ and keeps the data point and all other angles fixed.

A rotation gate. The rotation gates $R_x$, $R_y$ and $R_z$ have the form below. $P$ stands for one of three fixed 2 by 2 matrices, which are called $X$, $Y$ and $Z$. For $R_y$ it is $Y$. (The letter is the slide's. This $P$ is not the phase gate P of the feature map drawing, and it is not a probability.) All that matters here is a property the three share: applied twice, each gives the identity $I$, the matrix that changes nothing.

$$R(\theta) = e^{-i\theta P/2}, \quad P^2 = I$$

The exponential of a matrix is defined by the same endless sum of powers as the exponential of a number:

$$e^{-i\theta P/2} = I + \left( -\frac{i\theta}{2} \right) P + \frac{1}{2!}\left( -\frac{i\theta}{2} \right)^2 P^2 + \frac{1}{3!}\left( -\frac{i\theta}{2} \right)^3 P^3 + \frac{1}{4!}\left( -\frac{i\theta}{2} \right)^4 P^4 + \cdots$$

Because $P^2 = I$, every even power of $P$ is $I$ and every odd power is $P$. Because $i^2 = -1$, the powers of $-\frac{i\theta}{2}$ are $\left( -\frac{i\theta}{2} \right)^2 = -\left( \frac{\theta}{2} \right)^2$, $\left( -\frac{i\theta}{2} \right)^3 = i\left( \frac{\theta}{2} \right)^3$, $\left( -\frac{i\theta}{2} \right)^4 = \left( \frac{\theta}{2} \right)^4$, and so on. Collect the terms with $I$ and the terms with $P$:

$$e^{-i\theta P/2} = \left( 1 - \frac{1}{2!}\left( \frac{\theta}{2} \right)^2 + \frac{1}{4!}\left( \frac{\theta}{2} \right)^4 - \cdots \right) I - i\left( \frac{\theta}{2} - \frac{1}{3!}\left( \frac{\theta}{2} \right)^3 + \frac{1}{5!}\left( \frac{\theta}{2} \right)^5 - \cdots \right) P$$

The first bracket is the endless sum that defines $\cos\frac{\theta}{2}$, and the second is the one that defines $\sin\frac{\theta}{2}$. So the gate is a mix of "do nothing" and "apply $P$", with weights $\cos\frac{\theta}{2}$ and $\sin\frac{\theta}{2}$.

$$R(\theta) = \cos\frac{\theta}{2}\, I - i \sin\frac{\theta}{2}\, P$$

Its expectation value is a sinusoid, which is a wave with the shape of a sine curve. After the gate, every amplitude of the state is a sum of two parts, one with the factor $\cos\frac{\theta}{2}$ and one with the factor $\sin\frac{\theta}{2}$. An expectation value multiplies amplitudes in pairs, so $f$ contains only the products $\cos^2\frac{\theta}{2}$, $\sin^2\frac{\theta}{2}$ and $\sin\frac{\theta}{2}\cos\frac{\theta}{2}$. Three identities from trigonometry turn these into a constant, a cosine and a sine of the full angle:

$$\cos^2\frac{\theta}{2} = \frac{1 + \cos\theta}{2}, \qquad \sin^2\frac{\theta}{2} = \frac{1 - \cos\theta}{2}, \qquad \sin\frac{\theta}{2}\cos\frac{\theta}{2} = \frac{\sin\theta}{2}$$

So, as a function of this one angle, $f$ is a constant plus a cosine plus a sine. The numbers $a$, $b$ and $c$ do not depend on $\theta$. They depend on everything else in the circuit.

$$f(\theta) = a + b\cos\theta + c\sin\theta$$

Exact gradient from two runs. The symbol on the left is the slope of $f$ with respect to $\theta$ when nothing else changes. To get it, evaluate $f$ with the angle shifted up by a quarter turn and down by a quarter turn, and halve the difference.

$$\frac{\partial f}{\partial\theta} = \frac{f\left( \theta + \frac{\pi}{2} \right) - f\left( \theta - \frac{\pi}{2} \right)}{2}$$

Check it on the cosine term. The slope of $b\cos\theta$ is $-b\sin\theta$. Use $\cos\left( \theta + \frac{\pi}{2} \right) = -\sin\theta$ and $\cos\left( \theta - \frac{\pi}{2} \right) = \sin\theta$.

$$\frac{b\cos\left( \theta + \frac{\pi}{2} \right) - b\cos\left( \theta - \frac{\pi}{2} \right)}{2} = \frac{-b\sin\theta - b\sin\theta}{2} = -b\sin\theta$$

The other two terms work the same way. The slope of $c\sin\theta$ is $c\cos\theta$, and $\sin\left( \theta \pm \frac{\pi}{2} \right) = \pm\cos\theta$.

$$\frac{c\sin\left( \theta + \frac{\pi}{2} \right) - c\sin\left( \theta - \frac{\pi}{2} \right)}{2} = \frac{c\cos\theta + c\cos\theta}{2} = c\cos\theta$$

The constant $a$ has slope 0, and it cancels in the difference. So the rule gives the exact slope of all three terms, and therefore of $f$.

What this buys. The gradient costs two extra evaluations of the circuit for each angle, and the result is exact. The usual numerical way to get a slope, called a finite difference, is to shift the angle by a tiny step $h$ in both directions and divide the change in $f$ by $2h$. That is only an approximation, and on hardware it is worse: the tiny change in $f$ drowns in shot noise. The parameter-shift rule compares two values of $f$ that are far apart.

The cell below computes the slope of $f$ with respect to the first of the 12 angles, for the first training point and the random starting angles, in both ways.

In [ ]:
which = 0                                  # which of the 12 angles to look at
one_point = X_train[:1]                    # the first training point

shift = np.zeros(len(theta_start))
shift[which] = np.pi / 2                   # a quarter turn on that angle, nothing on the others
by_shift = (f(one_point, theta_start + shift)[0] - f(one_point, theta_start - shift)[0]) / 2

tiny = np.zeros(len(theta_start))
tiny[which] = 1e-6                         # a tiny step on that angle
by_difference = (f(one_point, theta_start + tiny)[0] - f(one_point, theta_start - tiny)[0]) / 2e-6

print("slope from the parameter-shift rule:", round(float(by_shift), 6))
print("slope from a finite difference:     ", round(float(by_difference), 6))

Both give $-0.260397$. On a simulator without shot noise the finite difference is good too. On hardware only the first way is practical.

The last cell of Part 1 makes the second step of the derivation visible. It turns the same angle through a full circle in 25 steps and plots $f$ at each step. Three of the 25 values, at 0, $\frac{\pi}{2}$ and $\pi$, are enough to find $a$, $b$ and $c$. The gray curve is the sinusoid with these three numbers, and the red line has the slope that the parameter-shift rule gave.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 25)                 # 25 values of the angle, from 0 to 2 pi
values = []
for angle in angles:
    theta = theta_start.copy()
    theta[which] = angle                               # change one angle, keep the other 11
    values.append(f(one_point, theta)[0])
values = np.array(values)

a = (values[0] + values[12]) / 2                       # values[0] is f at angle 0, values[12] is f at angle pi
b = (values[0] - values[12]) / 2
c = values[6] - a                                      # values[6] is f at angle pi/2
print("all 25 values lie on a + b cos + c sin:", np.allclose(values, a + b * np.cos(angles) + c * np.sin(angles)))

here = theta_start[which] % (2 * np.pi)                # the starting value of this angle
f_here = a + b * np.cos(here) + c * np.sin(here)
smooth = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(smooth, a + b * np.cos(smooth) + c * np.sin(smooth), color="0.5", label="a + b cos(theta) + c sin(theta)")
ax.plot(angles, values, "o", color="tab:blue", label="f from the circuit")
ax.plot([here - 0.6, here + 0.6], [f_here - 0.6 * by_shift, f_here + 0.6 * by_shift], color="tab:red", lw=2,
        label="slope from the parameter-shift rule")
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax.set_xticklabels(["0", "pi/2", "pi", "3 pi/2", "2 pi"])
ax.set_xlabel("one angle of the trainable block")
ax.set_ylabel("f")
ax.set_title("f as one angle changes")
ax.legend(fontsize=8)
plt.show()

All 25 values from the circuit lie on the sinusoid, and the red line touches the curve with the right slope.

### An honest look at quantum machine learning

*Slide 26.*

- Kernels are well defined and easy to test.
- Proven advantages exist only for specially constructed problems.
- On everyday data sets, classical methods are still the ones to beat.

Put plainly: a quantum kernel is a precise mathematical object, and you can test a kernel classifier in an afternoon, as tonight's lab does. What is missing is a reason to expect it to win. Where a quantum kernel has been proven to beat classical methods, the problem was built for that purpose, and even there the proof rests on an assumption: in the best-known case, that a certain problem about whole numbers cannot be solved quickly on an ordinary computer. On data sets like tonight's, a well-chosen classical method is at least as good and far cheaper. Lab 3 makes the comparison on tonight's data, with the same training and test points.

Variational classifiers have a difficulty of their own. As such circuits grow, the slopes that training relies on tend to shrink toward zero for most choices of the angles, and the optimizer has nothing to follow. These vanishing gradients are known as barren plateaus.

So the value of tonight is not a faster classifier. It is knowing the workflow, from data to feature map to kernel to score, and being able to judge a claim when you read one.

The picture on the slide makes this comparison for tonight's data. It shows the data twice, once for the lab's quantum kernel and once for a classical kernel. The colored areas are where each classifier predicts label 0 and label 1, the dots are the 60 test points, and a ring marks a point that is labeled wrong: eight for the quantum kernel, three for the classical one. Lab 3 prints both scores.

## Part 2: Hands-on

The code in the "Lab" cells is the code on the slides. "File on the slide" gives the name shown above that code on the slide; there is no separate file to download. Run each cell, read what it prints, then change something and run it again. After each lab comes a "Your turn" exercise with a check cell.

### Lab 1: where does your simulator give up?

*Slide 31.*

File on the slide: `scaling.py`

1. Simulate a GHZ circuit at growing $n$
2. Time the state-vector method
3. Switch to matrix product state
4. Compare the two

This lab puts a stopwatch on the two ways of storing a state from Part 1. Qiskit Aer is the simulator package of Qiskit, and `AerSimulator` is its simulator. With `method="statevector"` it stores all $2^n$ amplitudes. With `method="matrix_product_state"` it stores a matrix product state. The cell runs the GHZ circuit on 10, 16, 20 and 24 qubits with each method, 100 shots each time. Line by line:

- `transpile(ghz(n), sim)` rewrites the circuit for the simulator `sim`. To transpile a circuit is to rewrite it with the gates that one particular backend understands, and a backend is the machine or simulator that runs a circuit. Aer understands H and CX as they are, so there is little to rewrite here.
- `time.perf_counter()` reads a stopwatch. The difference between two readings, `dt`, is the time in seconds that the run took.
- `sim.run(qc, shots=100).result()` runs the circuit and waits until the result is there.
- The `print` line shows the method, the number of qubits and the time, rounded to two decimals.

The last line of the slide's code says that `ghz(n)` is Monday's function. It comes from Lab 3 of Session 1. The small cell before the lab code defines it again, so that this notebook does not need Monday's.

In [ ]:
# Glue for the next cell: ghz(n) is Monday's function, from Lab 3 of Session 1.
# It builds the GHZ circuit on n qubits: one H, a chain of CX gates, then a measurement of every qubit.
from qiskit import QuantumCircuit


def ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)   # one more qubit joins
    qc.measure_all()
    return qc

In [ ]:
import time
from qiskit import transpile
from qiskit_aer import AerSimulator

for method in ("statevector",
               "matrix_product_state"):
    sim = AerSimulator(method=method)
    for n in (10, 16, 20, 24):
        qc = transpile(ghz(n), sim)
        t0 = time.perf_counter()
        sim.run(qc, shots=100).result()
        dt = time.perf_counter() - t0
        print(method, n, round(dt, 2), "s")

# ghz(n) is Monday's function

The cell prints eight lines, four sizes for each method. The times depend on your computer, so they differ from everyone else's and from one run to the next. What to look for:

- **State vector.** The time grows with the number of qubits. Because the slide's code rounds to two decimals (`round(dt, 2)`), the smallest sizes can show `0.0 s`, and timings of a few hundredths of a second are noisy. The growth is plain from 20 qubits to 24: four more qubits are $2^4 = 16$ times as many amplitudes. On the laptop used to test this notebook the 24-qubit run took about 0.3 seconds when the laptop was otherwise idle and more than a second when it was busy.
- **Matrix product state.** All four lines stay small, `0.0 s` or a few hundredths of a second, and they do not grow in step with the number of qubits. For a GHZ state this method hardly notices whether there are 10 qubits or 24.

The next cell is not on the slide. It makes the doubling visible. It runs the state-vector method on 24, 25 and 26 qubits and prints each time with three decimals, the memory that the state vector takes, and how the time compares with the line before. The largest run needs 1 GiB of memory. If your computer is short of memory, change the 26 to 25 before you run the cell. The cell takes a few seconds on a recent laptop and longer on a slower computer.

In [ ]:
exact = AerSimulator(method="statevector")
previous = None
for size in (24, 25, 26):
    circuit = transpile(ghz(size), exact)
    start = time.perf_counter()
    result = exact.run(circuit, shots=100).result()
    seconds = time.perf_counter() - start
    if not result.success:                     # the simulator refused, for example for lack of memory
        print(size, "qubits: the run failed:", result.status)
        break
    line = f"{size} qubits: {seconds:.3f} s | state vector: {16 * 2 ** size // 2 ** 20:,} MiB"
    if previous is not None:
        line += f" | {seconds / previous:.1f} times the time of the line before"
    print(line)
    previous = seconds

Each added qubit doubles the memory exactly: 256 MiB, 512 MiB, 1,024 MiB. The time roughly doubles as well, so expect ratios near 2. Single timings are noisy, because other programs on your computer run at the same time and take part of its computing power. In the test runs for this notebook, on a busy laptop, the ratios ranged from 1.4 to 3.3. Run the cell a second time and compare.

To find where your own computer gives up, raise the sizes one qubit at a time and stop when a run takes longer than you want to wait. Do the arithmetic before you run: 27 qubits need 2 GiB of memory, 28 need 4 GiB, 29 need 8 GiB and 30 need 16 GiB. Stop while the state vector is still well below the memory of your computer: at 27 or 28 on a laptop with 16 GiB, and one qubit earlier on a laptop with 8 GiB. On the free Google Colab server, which has about 12 GB of memory, stop at 27. If a server runs out of memory, the session restarts and every variable is lost: run the notebook again from Setup. Aer may not stop you. It accepts every run whose state vector is no larger than the whole memory of the computer, although other programs already use part of that memory. In a test for this notebook, on a laptop with 16 GiB of memory, it accepted a 30-qubit run and began to fill the memory. Past the size that your computer's memory can hold, the `transpile` line stops with an error named `CircuitTooWideForTarget`.

Step 4 of the slide: compare the two methods. The state-vector method pays for $2^n$ amplitudes, whatever the state is. The matrix-product-state method pays for entanglement, and a GHZ state has little of it: bonds of size 2 are enough, as Part 1 showed. That is why its lines hardly move. GHZ is the friendly case for a tensor network. A circuit that entangles its qubits strongly needs large bonds, and the advantage is gone.

#### Your turn 1

Simulate the GHZ circuit on 60 qubits, `ghz(60)`, with 100 shots, and store the counts in `counts60`. By the formula from Part 1, a state vector of 60 qubits would need $2^{60} \times 16$ bytes, about a billion times the 16 GiB of 30 qubits, so choose your method accordingly. A finished run hands you its counts with `.result().get_counts()`.

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

counts60 = None
# YOUR CODE HERE
print(counts60)

In [ ]:
assert counts60 is not None, "counts60 is still None. Which of the lab's two methods does not need 2**60 amplitudes? Run ghz(60) with it."
assert isinstance(counts60, dict) and len(counts60) > 0, "counts60 should be the dictionary of counts that .result().get_counts() returns."
assert all(isinstance(outcome, str) and len(outcome) == 60 for outcome in counts60), "Every outcome should have 60 characters, one for each qubit. Run the circuit ghz(60)."
assert sum(counts60.values()) == 100, "The task asks for 100 shots."
assert set(counts60) == {"0" * 60, "1" * 60}, "A GHZ state gives all zeros and all ones, each in about half of the shots, and nothing else. Run the circuit that ghz(60) returns, without changing it."
print("Check passed: 60 qubits, and only all zeros and all ones appear.")
print("shots that read all zeros:", counts60.get("0" * 60, 0), "| shots that read all ones:", counts60.get("1" * 60, 0))

Sixty qubits ran in about the time that ten did. A state vector of that size is far beyond the memory of any computer, and yet this particular state is easy, because it has so little entanglement. Keep the other half of the rule in mind as well: a simulator that is fast on one circuit says nothing about a circuit with a lot of entanglement.

### Lab 2: build a kernel matrix

*Slide 32.*

File on the slide: `kernel.py`

1. Load the small data set
2. Write the feature map
3. Compute $k$ for every training pair
4. Plot the matrix

Step 1. The small data set is the one from Part 1: 100 points in two interleaved half-moons, each with two features and a label 0 or 1. 40 points are for training and 60 for testing. The features are rescaled to angles between 0 and $\frac{\pi}{2}$, with the scale set by the training points alone. The slide's code expects the data under the names `X_train`, `X_test`, `y_train` and `y_test`. The cell below makes it again, with the same lines as in Part 1, so the lab also runs if you skipped Part 1 or changed the data there.

In [ ]:
# Glue for the next cell: the small data set, made with the same lines as in Part 1.
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X, y = make_moons(n_samples=100, noise=0.15, random_state=2026)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=40, stratify=y, random_state=2026)
scaler = MinMaxScaler(feature_range=(0, np.pi / 2)).fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
print("training points:", len(X_train), "| test points:", len(X_test))

Steps 2 and 3 are the lab code. Line by line:

- `fmap = zz_feature_map(2, reps=2)` is the ZZ feature map from Part 1 for two features, with its block of gates twice.
- `state(x)` puts one data point into the feature map with `assign_parameters` and returns its feature state as a `Statevector`.
- `kernel(A, B)` takes two lists of data points. It makes the feature state of every point and returns a table of kernel values. The entry in row $i$ and column $j$ compares point $i$ of `A` with point $j$ of `B`: `a.inner(b)` is the overlap of the two states, and `abs(...) ** 2` is its squared size.
- `K_train = kernel(X_train, X_train)` is the kernel matrix of the training set, with 40 rows and 40 columns.
- `K_test = kernel(X_test, X_train)` compares every test point with every training point. It has 60 rows, one for each test point, and 40 columns, one for each training point. Lab 3 needs it to label the test points.

On a simulator the overlap is read straight from the state vectors. On hardware every entry would come from its own compute-and-uncompute circuit, as in Part 1: 780 circuits for `K_train` and 2,400 for `K_test`.

The cell prints nothing. The cells after it look at the result.

In [ ]:
import numpy as np
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector

fmap = zz_feature_map(2, reps=2)

def state(x):
    return Statevector(fmap.assign_parameters(x))

def kernel(A, B):
    sa = [state(x) for x in A]
    sb = [state(x) for x in B]
    return np.array([[abs(a.inner(b)) ** 2
                      for b in sb] for a in sa])

K_train = kernel(X_train, X_train)
K_test = kernel(X_test, X_train)

The next cell checks the two matrices against the "What you should see" table of the slides: the kernel matrix of the training set should be symmetric, which means that the entry in row $i$ and column $j$ equals the entry in row $j$ and column $i$, and it should have ones on the diagonal.

In [ ]:
print("K_train:", K_train.shape[0], "rows and", K_train.shape[1], "columns")
print("K_test: ", K_test.shape[0], "rows and", K_test.shape[1], "columns")
print("K_train is symmetric:", np.allclose(K_train, K_train.T))
print("the diagonal of K_train is all ones:", np.allclose(np.diag(K_train), 1))
print("smallest entry of K_train:", round(float(K_train.min()), 4), "| largest:", round(float(K_train.max()), 4))

Both checks print `True`. `np.allclose` compares numbers and ignores rounding errors in the last decimals. That matters here. In exact arithmetic every diagonal entry is 1. The computer rounds every number it stores, so a computed entry can miss 1 by about $10^{-15}$, in the fifteenth or sixteenth decimal. All entries lie between 0 and 1, as squared overlaps must.

Step 4 of the slide is to plot the matrix. The slide has no code for it, so the cell below is an addition. It draws the kernel matrix as a heat map: one small square for each entry, bright for a value near 1 and dark for a value near 0. Before drawing, it puts the training points in a new order, all points with label 0 first and then all points with label 1. Without that the picture looks like noise. The cell also prints the average kernel value for pairs of points with the same label and for pairs with different labels.

In [ ]:
import matplotlib.pyplot as plt

order = np.argsort(y_train, kind="stable")      # positions of the label-0 points, then of the label-1 points
K_sorted = K_train[order][:, order]             # the same matrix with rows and columns in that order
same_0 = K_sorted[:20, :20]                     # label 0 with label 0
same_1 = K_sorted[20:, 20:]                     # label 1 with label 1
different = K_sorted[:20, 20:]                  # label 0 with label 1
# In the two blocks with equal labels, leave out the 20 ones on the diagonal (a point with itself).
print("average kernel value, two different points with label 0:", round(float((same_0.sum() - 20) / 380), 3))
print("average kernel value, two different points with label 1:", round(float((same_1.sum() - 20) / 380), 3))
print("average kernel value, two points with different labels: ", round(float(different.mean()), 3))

fig, ax = plt.subplots(figsize=(4.8, 4))
image = ax.imshow(K_sorted, cmap="viridis", vmin=0, vmax=1)
ax.axhline(19.5, color="white", lw=1)           # the white lines separate the two labels
ax.axvline(19.5, color="white", lw=1)
ax.set_xticks([9.5, 29.5])
ax.set_xticklabels(["label 0", "label 1"])
ax.set_yticks([9.5, 29.5])
ax.set_yticklabels(["label 0", "label 1"], rotation=90, va="center")
ax.set_title("Kernel matrix, sorted by label")
fig.colorbar(image, ax=ax, label="kernel value")
plt.show()

The bright line from the top left to the bottom right is the diagonal: every point compared with itself gives 1. The picture is its own mirror image across that line, which is the symmetry. The white lines divide the matrix into four blocks. If the kernel judged "alike" the way the labels do, the two blocks on the diagonal would be bright and the other two dark. The real picture is less clear-cut. Two points with the same label have an average kernel value of 0.506 or 0.548, and two points with different labels have 0.38. The difference is there, and it is modest. The classifier in Lab 3 has to work with it.

The slide "The plan: compare every pair, then classify" shows this matrix in other colors: there, dark blue means a value near 1.

#### Your turn 2

Part 1 showed that the kernel value of two points is the probability of all zeros at the end of a compute-and-uncompute circuit. Build that circuit for the first two training points, with the lab's feature map `fmap`, and call it `cu`: first the feature map for `X_train[0]`, then the inverse of the feature map for `X_train[1]`. Do not add measurements.

Three tools are enough. `fmap.assign_parameters(x)` is the feature map with the data point `x` filled in. `a.compose(b)` is the circuit that runs `a` and then `b`. `b.inverse()` is the circuit that undoes `b`.

In [ ]:
cu = None
# YOUR CODE HERE

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

assert cu is not None, "cu is still None. Build the circuit: the feature map for X_train[0], then the inverse of the feature map for X_train[1]."
assert isinstance(cu, QuantumCircuit), "cu should be a circuit. assign_parameters(), compose() and inverse() each return one."
assert cu.num_qubits == 2 and cu.num_parameters == 0, "cu should have two qubits and no open parameters. Fill in both data points with assign_parameters."
body = cu.remove_final_measurements(inplace=False)
try:
    p_zeros = float(abs(Statevector(body).data[0]) ** 2)
except Exception:
    p_zeros = None      # for example a measurement in the middle of the circuit
assert p_zeros is not None, "cu should hold gates only. Take out any measurement that is not at the very end of the circuit."
assert abs(p_zeros - K_train[0, 1]) < 1e-9, "The probability of all zeros at the end of cu is not K_train[0, 1]. Use the lab's feature map fmap, not demo_map from Part 1, and the training points X_train[0] and X_train[1]: run the map for one point, then undo the map for the other."
print("Check passed.")
print("probability of 00 at the end of cu:", round(p_zeros, 6))
print("the kernel entry K_train[0, 1]:    ", round(float(K_train[0, 1]), 6))
cu.draw("mpl", scale=0.7)

### Lab 3: train, test, and score

*Slide 33.*

File on the slide: `train.py`

1. Pass the kernel matrix to a support vector classifier
2. Predict on the test split
3. Report accuracy

The quantum part of the work is done. What is left is classical, and scikit-learn does it. Line by line:

- `SVC` is scikit-learn's support vector classifier. `kernel="precomputed"` tells it that it will be handed kernel values, not features.
- `clf.fit(K_train, y_train)` is the training. The classifier gets the kernel matrix of the 40 training points and their 40 labels.
- `clf.score(K_test, y_test)` labels the 60 test points from their kernel values with the training points, compares the predictions with the true labels, and returns the fraction that is right. That fraction is the test accuracy. The test split on the slide is the test set.
- The two comment lines at the end are the slide's ideas for improving the score. A cell further down tries some of them.

In [ ]:
from sklearn.svm import SVC

clf = SVC(kernel="precomputed")
clf.fit(K_train, y_train)

accuracy = clf.score(K_test, y_test)
print("test accuracy:", accuracy)

# ideas: change reps, rescale the features,
# try another feature map, tune C

The cell prints `test accuracy: 0.8666666666666667`. The classifier labels 52 of the 60 test points correctly. Nothing in Labs 2 and 3 is random, so you should see the same number. Note it: this is tonight's baseline. Challenge 4 uses the same code on a data set of its own.

`clf.score` does steps 2 and 3 of the slide in one call. The next cell does step 2 separately, so that you can see the predictions. It is not on the slide.

In [ ]:
test_pred = clf.predict(K_test)          # the predicted label of every test point
print("predicted labels of the first ten test points:", test_pred[:10])
print("true labels of the first ten test points:     ", y_test[:10])
print("correct predictions:", int((test_pred == y_test).sum()), "of", len(y_test))

The first ten predictions are all correct. Over the whole test set, 52 of 60 are.

Now the ideas from the slide's comment: change `reps`, rescale the features, tune `C`. `C` is a setting of the support vector classifier. It says how hard the classifier tries to get every single training point right: a larger `C` tolerates fewer mistakes on the training set. The cell below repeats Labs 2 and 3 with one change at a time and prints the test accuracy. The last line is the comparison that the "honest look" slide asks for: a classical support vector classifier on the same training and test points. Its kernel, called `rbf`, is a standard classical one, computed directly from the features without any circuit.

In [ ]:
def quantum_accuracy(reps=2, C=1.0, stretch=1.0):
    """Repeat Labs 2 and 3 with other settings and return the test accuracy.

    reps: repetitions of the feature map. C: the setting of the classifier.
    stretch: every feature is multiplied by this number before it becomes an angle.
    """
    other_map = zz_feature_map(2, reps=reps)

    def other_kernel(A, B):
        sa = [Statevector(other_map.assign_parameters(stretch * x)) for x in A]
        sb = [Statevector(other_map.assign_parameters(stretch * x)) for x in B]
        return np.array([[abs(a.inner(b)) ** 2 for b in sb] for a in sa])

    other_clf = SVC(kernel="precomputed", C=C).fit(other_kernel(X_train, X_train), y_train)
    return other_clf.score(other_kernel(X_test, X_train), y_test)


print("the lab's settings (reps=2, C=1, features from 0 to pi/2):", round(quantum_accuracy(), 4))
print("reps=1:                                                  ", round(quantum_accuracy(reps=1), 4))
print("C=10:                                                    ", round(quantum_accuracy(C=10), 4))
print("features from 0 to pi (every feature doubled):           ", round(quantum_accuracy(stretch=2), 4))
print("features from 0 to 2 pi (every feature times four):      ", round(quantum_accuracy(stretch=4), 4))

classical = SVC(kernel="rbf").fit(X_train, y_train)          # a classical kernel, straight from the features
print("classical kernel (rbf), same training and test points:   ", round(classical.score(X_test, y_test), 4))

Read the six lines.

- One repetition of the feature map is a little worse than two: 0.8333 against 0.8667.
- `C=10` helps: 0.9333.
- The scaling of the features matters most. With the features stretched to the range from 0 to $\pi$ the accuracy falls to 0.6333, and from 0 to $2\pi$ it is 0.5167, which is no better than guessing. This is the row "Accuracy stuck near one half" of the troubleshooting table below.
- The classical kernel scores 0.95, which is 57 of the 60 test points. That is five more than the quantum kernel with the lab's settings, with no circuit at all.

The last line, put plainly: on this data set the classical method wins, and it is cheaper. That is what the slide "An honest look at quantum machine learning" means by "the ones to beat".

One caution about the lines in the middle. Each of them was judged on the test set. If you try many settings and keep the one with the best test accuracy, that accuracy flatters you, because the choice itself has used the test points. A fair final score needs points that played no part in any choice. Such points are called a held-out test set.

#### Your turn 3

Classify one point from scratch. `x_new` is a point to classify. Here it is a copy of one of the test points, `X_test[16]`. First build `k_new`, its kernel values with all 40 training points, as a table with one row and 40 columns. Then store the label that the trained classifier `clf` predicts for it in `label_new`.

`kernel()` from Lab 2 and `clf.predict()` from this lab are all you need. Remember that `kernel()` expects two lists of points, and that `clf.predict()` returns a list of labels, one for each row it is given.

In [ ]:
x_new = X_test[16].copy()        # the point to classify
k_new = label_new = None
# YOUR CODE HERE
print("predicted label:", label_new)

In [ ]:
assert k_new is not None, "k_new is still None. First build the kernel values of x_new with every training point. If the cell above stopped with a TypeError, kernel() was given a single point where it takes a list of points."
assert np.asarray(k_new).shape == (1, len(X_train)), "k_new needs one row, and one column for each training point. kernel() takes two lists of points: first a list that holds only x_new, then X_train. Keep the whole table that kernel() returns."
assert np.allclose(np.asarray(k_new)[0], K_test[16]), "These are not the kernel values of x_new. Call kernel() with the new point first and the training set second."
assert label_new is not None, "label_new is still None. Let clf predict a label from the kernel values. If the cell above stopped with a ValueError about features, clf.predict() was given something other than a table with one column for each training point."
assert np.ndim(label_new) == 0, "label_new should be a single label, not a list of labels. clf.predict() returns one label for each row it is given."
assert label_new == clf.predict(K_test[16:17])[0], "This is not the label that clf predicts for this point. The task asks for the prediction of the classifier, which it makes from the kernel row, not from the point itself and not from y_test."
print("Check passed.")
print("predicted label:", label_new, "| true label of this test point:", y_test[16])

This is the whole pipeline of a quantum kernel classifier, for one point: a feature map turns the point into a state, overlaps with the training states give a row of kernel values, and a classical classifier turns that row into a label. Challenge 4 asks you to make this pipeline as accurate as you can, on a data set of its own.

## What you should see

*Slide 34.*

| Lab | Expected result |
|---|---|
| Lab 1: scaling test | State-vector time roughly doubles per qubit; matrix product state barely moves for GHZ |
| Lab 2: kernel matrix | A symmetric matrix with ones on the diagonal |
| Lab 3: classifier | A test accuracy between 0 and 1; note yours as the baseline |

Two notes on these rows. In Lab 1 the doubling shows only from about 20 qubits up. Below that a run is over in a few hundredths of a second and the two decimals of the slide's code hide the growth, which is why the lab has the extra cell for 24, 25 and 26 qubits. In Lab 3 the lab's settings give a test accuracy of 0.8667, which is 52 of the 60 test points.

## If something breaks

*Slide 35.*

| What you see | What to do |
|---|---|
| The Jupyter kernel dies or runs out of memory | Lower the qubit count: each qubit doubles the memory |
| `assign_parameters` complains about length | The feature map takes one value per feature, two here |
| The classifier rejects the kernel shape | The test kernel needs test rows and training columns |
| Accuracy stuck near one half | Rescale the features before encoding them as angles |
| `CircuitTooWideForTarget` in Lab 1 | The circuit has more qubits than this simulator method accepts. For the state-vector method that is what fits in your computer's memory: lower the qubit count, or use the matrix-product-state method. For the matrix-product-state method the ceiling is 63 qubits |

How these rows look in this notebook, on the tested versions:

- The Jupyter kernel in the first row is the Python process that runs the cells. It is not the quantum kernel of Lab 2. If it dies, restart it (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. Google Colab: Runtime menu, Restart session) and run the notebook again from the top with a smaller number of qubits.
- In the second row the message starts with `ValueError: Mismatching number of values and parameters.` It appears when `state()` or `assign_parameters()` gets a data point with one number, or with three or more. A single number that is not inside a list gives a different message, `TypeError: 'numpy.float64' object is not iterable`. That one appears when `kernel()` is handed one point where it expects a list of points.
- In the third row the message is `ValueError: X has 60 features, but SVC is expecting 40 features as input.` scikit-learn calls the columns of the table "features". The table handed to `clf.score` or `clf.predict` must have one column for each training point: `kernel(X_test, X_train)`, not the other way around.
- The fourth row is what the Lab 3 cell with the six accuracies showed: with the features stretched to the range from 0 to $2\pi$ the test accuracy was 0.5167.
- The last row is not on the slide. It is the error that `transpile` gives in Lab 1 when the state-vector method is asked for more qubits than fit in memory. The matrix-product-state method has a fixed ceiling in its place: `transpile` accepts at most 63 qubits for it, so `ghz(64)` stops with the same error although memory is not the problem. Without the `transpile` line, the matrix-product-state method runs `ghz(100)` too.

## Check yourself

*Slide 27.* On the slides these questions come before the labs.

1. How much memory does a 31-qubit state vector need?
2. What is the kernel entry $k(x, x)$ of a point with itself?
3. How many circuits does a kernel matrix on $N = 20$ training points need?

## Tonight in one slide

*Slide 36.*

- **Simulate.** Every amplitude is stored. The wall is near 50 qubits.
- **Scale up.** A thousand times the memory adds ten qubits.
- **Compress.** Tensor networks are cheap when entanglement is low.
- **Combine.** Each machine does what it is good at.
- **Learn.** A quantum kernel is an overlap you measure on a circuit.

Honesty: on everyday data sets, classical methods are still the ones to beat.

## Challenge 4: Quantum kernel classifier

*Slide 37.*

Reach the highest accuracy on a held-out test set. The challenge opens Thu, Nov 19 at 6:00 PM and closes Sun, Nov 22 at 11:59 PM, Clemson time (US Eastern).

Start from tonight's notebook: Labs 2 and 3 are the starting point. The challenge notebook is posted on the event page when the challenge opens, under Challenges, "Quantum kernel classifier". Submission steps are in the challenge notebook.

## Where to go from here

*Slides 38 and 39.*

- **Tonight.** Beat tonight's baseline, then start Challenge 4.
- **Try it.** Change the repetitions or the feature scaling.
- **Stuck?** Ask in Discord between sessions.

To change the repetitions, change `reps=2` in the line `fmap = zz_feature_map(2, reps=2)` of Lab 2, then run Labs 2 and 3 again. To change the feature scaling, change `feature_range=(0, np.pi / 2)` in the first code cell of Lab 2, the one that makes the data set, then run the cells from there on.

A quantum computer is one part of a bigger machine.

- Slides and Challenge 4 are posted tonight.
- Four challenges are open until Sunday.
- Tomorrow at 5:00 PM: Session 5, real hardware, noise, and error correction.